# Research Notebook 04: Machine Learning Volatility Prediction
## Linear Regularization, Tree Ensembles, Gradient Boosting & Uncertainty

### 1. Research Objective
Can machine learning models with rich technical, range-based, and distributional features outperform traditional econometric baselines?


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
from volatility_lab.data.providers.synthetic import SyntheticMarketDataProvider
from volatility_lab.data.preprocessing import DataPreprocessor
from volatility_lab.features.feature_pipeline import FeaturePipeline
from volatility_lab.machine_learning.registry import MLModelRegistry
from volatility_lab.evaluation.metrics import compute_all_metrics

raw = SyntheticMarketDataProvider(seed=42).fetch_data('NIFTY50', '2019-01-01', '2023-12-31')
df = DataPreprocessor().clean_and_prepare(raw)

pipe = FeaturePipeline()
X, y = pipe.transform(df, horizon=5)

split = int(len(X) * 0.75)
X_tr, y_tr = X.iloc[:split], y.iloc[:split]
X_te, y_te = X.iloc[split:], y.iloc[split:]

models = ['ridge', 'random_forest', 'lightgbm']
results = []

for m_name in models:
    m = MLModelRegistry.get(m_name)
    m.fit(X_tr, y_tr)
    preds = np.maximum(m.predict(X_te), 1e-6)
    metrics = compute_all_metrics(y_te.values, preds)
    metrics['model'] = m_name
    results.append(metrics)

perf_df = pd.DataFrame(results).set_index('model')
print(perf_df[['qlike', 'rmse', 'mae', 'r2']])


### 2. Takeaways & Feature Importance
- Gradient boosting captures non-linear interactions between range estimators and volume shifts.
- Tree models show superior stability during volatility spikes.
